# 07 · V3 RAW, V3 EMA và A1 trên HC3

Mỗi seed huấn luyện V3 một lần và A1 một lần. RAW là trọng số được optimizer cập nhật. EMA lấy trung bình trượt của trọng số RAW sau mỗi bước. Mỗi bộ trọng số V3 chọn epoch và ngưỡng human95 riêng trên HC3 validation. Hai phép ghép RAW+A1 và EMA+A1 dùng chung điểm A1 và phân loại bằng tích P(human) ở ngưỡng 0,5. Notebook in accuracy trên HC3 test, GPT-3.5 Mixed sạch và HC3 Replace 25%. Bật GPU và Internet trên Kaggle.

Cột A1/V3 routed dùng A1 cho văn bản có cách dùng từ gần HC3 sạch và V3 RAW ở ngưỡng human95 khi phát hiện dấu hiệu thay từ.


In [ ]:
%pip install -q torch-geometric transformers==4.44.2 'spacy>=3.8,<3.9' scikit-learn tqdm
!python -m spacy download en_core_web_sm


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, random, zipfile, hashlib
from importlib.metadata import version
from tempfile import TemporaryDirectory
from time import perf_counter
import numpy as np
import torch

ROOT = Path('/kaggle/working/v3_a1_train')
REPO = ROOT / 'PRDetect-Research'
BRANCH = 'version3'
SEED = 2026
WORK = ROOT / f'dual_seed{SEED}'
FEATURE_CACHE = ROOT / 'feature_cache'
# Ở phiên Kaggle mới, đặt đường dẫn tới feature_cache trong output đã lưu.
CACHE_INPUT = None
PARALLEL_IF_TWO_GPUS = True
V3_EPOCHS = 10
BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1
VAL_BATCH_SIZE = 64
A1_EPOCHS = 10
A1_LR = 0.0001
A1_BATCH = 4
WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/tphan1420/PRDetect-Research.git', str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
assert torch.cuda.is_available(), 'Enable the Kaggle GPU accelerator'
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('GPU:', torch.cuda.get_device_name())


## Chuẩn bị graph và huấn luyện V3

Trong cùng phiên, đổi SEED rồi chạy lại từ cell cấu hình để dùng lại graph và embedding. Ở phiên Kaggle mới, gắn output đã lưu của lần trước và đặt CACHE_INPUT tới thư mục feature_cache trong output đó. Mỗi seed xuất ba checkpoint gồm V3 RAW, V3 EMA và A1. Nếu có hai GPU, hai nhánh train đồng thời ở cell huấn luyện A1.


In [ ]:
%%writefile train_v3_dual_best.py
"""Train V3 once and select RAW and EMA independently on HC3 validation."""
import copy
import gc
import hashlib
import json
import math
from pathlib import Path
import time

from evaluate_v3_quang import REPO, TRAINING_DOMAINS, environment, sha256, write_json
from train_v3 import (atomic_save, build_model, build_parser, configure_determinism,
                      effective_readout, load_graph, validate_args, validation_loss)


def signature(args, variant, seed, data_hashes):
    config = vars(args).copy()
    for key in ("variants", "seeds", "output_dir", "graph_dir", "skip_completed", "overwrite"):
        config.pop(key)
    config["effective_readout"] = effective_readout(args, variant)
    code = {str(p.relative_to(REPO)): sha256(p) for p in (
        Path(__file__), REPO / "train_v3.py", REPO / "model/QuangV3.py", REPO / "building_graph.py", REPO / "dep_vocab.py",
        REPO / "dep_vocab.json",
    )}
    record = dict(config=config, variant=variant, seed=seed, data=data_hashes,
                  code_sha256=code, environment=environment())
    digest = hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()
    return digest, record


def completed_run(paths, record_path, run_signature):
    if not record_path.is_file() or not all(path.is_file() for path in paths.values()):
        return False
    record = json.loads(record_path.read_text(encoding="utf-8"))
    return (record.get("complete") is True and record.get("run_signature") == run_signature
            and record.get("checkpoint_sha256") == {w: sha256(p) for w, p in paths.items()})


def train_one(args, variant, seed, train, val, run_signature, provenance):
    import numpy as np
    import torch
    import torch.nn.functional as F
    from model.QuangV3 import predict, set_seed
    from evaluate_v3_quang import score_probabilities

    device = torch.device(args.device)
    set_seed(seed)
    model = build_model(args, variant, train.R).to(device)
    teacher = copy.deepcopy(model).eval() if args.ema else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    steps_per_epoch = math.ceil(train.G / args.batch_size)
    total = steps_per_epoch * args.epochs
    warm = max(1, int(total * 0.05))
    schedule = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: (s + 1) / warm if s < warm else
        0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))) * 0.95 + 0.05,
    )
    amp = bool(args.amp) and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    gm = train.GF.float().mean(0).cpu()
    gs = train.GF.float().std(0).cpu().clamp_min(1e-6)
    ema_checkpoint_path = args.output_dir / f"v3_{variant}_ema_seed{seed}.pt"
    raw_checkpoint_path = args.output_dir / f"v3_{variant}_raw_seed{seed}.pt"
    record_path = args.output_dir / f"v3_{variant}_dual_seed{seed}.train.json"
    saved_args = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}
    saved_args.update(seeds=[seed], cons=0.0, readout=effective_readout(args, variant))
    history = []
    best_ema_score = best_raw_score = float("inf")
    best_ema_epoch = best_raw_epoch = 0
    bad = step = 0
    start = time.time()
    sampler_generator = torch.Generator(device="cpu").manual_seed(seed)
    print(f"{variant} seed={seed} | R={train.R} mode={args.struct_mode} | "
          f"effective batch={args.batch_size}, micro={args.micro_batch_size}", flush=True)

    for epoch in range(1, args.epochs + 1):
        model.train()
        permutation = torch.randperm(train.G, generator=sampler_generator)
        epoch_loss = 0.0
        for offset in range(0, train.G, args.batch_size):
            indices = permutation[offset:offset + args.batch_size]
            optimizer.zero_grad(set_to_none=True)
            for micro in indices.split(args.micro_batch_size):
                batch = train.batch(micro, device, gm, gs)
                weight = batch["B"] / indices.numel()
                branch_weight = 0.5 if args.aug != "none" else 1.0
                # Release clean activations BEFORE the augmented forward. The
                # effective step still averages all examples/branches exactly.
                with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                    clean = model(batch)
                    clean_loss = F.binary_cross_entropy_with_logits(clean.float(), batch["y"])
                scaler.scale(weight * branch_weight * clean_loss).backward()
                epoch_loss += branch_weight * float(clean_loss.detach()) * batch["B"]
                del clean, clean_loss
                if args.aug != "none":
                    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                        noisy = model(batch, aug_mode=args.aug)
                        noisy_loss = F.binary_cross_entropy_with_logits(noisy.float(), batch["y"])
                    scaler.scale(weight * 0.5 * noisy_loss).backward()
                    epoch_loss += 0.5 * float(noisy_loss.detach()) * batch["B"]
                    del noisy, noisy_loss
                del batch
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                schedule.step()
                step += 1
                if teacher is not None:
                    decay = min(args.ema_decay, (1 + step) / (10 + step))
                    with torch.no_grad():
                        for target, current in zip(teacher.parameters(), model.parameters()):
                            target.mul_(decay).add_(current.detach(), alpha=1 - decay)
        # Each weight family chooses its own best validation epoch.  The raw
        # model and EMA teacher therefore receive identical training updates,
        # but neither is selected using the other model's validation scores.
        ema_model = teacher if teacher is not None else model
        raw_model = model
        truth = val.y.numpy().astype(int)

        def validate(candidate):
            probabilities = predict(candidate, val, device, gm, gs, bs=args.val_batch_size, amp=amp)
            swapped = predict(candidate, val, device, gm, gs, bs=args.val_batch_size, amp=amp, aug_mode="swap")
            metrics = score_probabilities(truth, probabilities, 0.5)
            clean_loss = validation_loss(probabilities, truth)
            swap_loss = validation_loss(swapped, truth)
            score = {"lossaug": clean_loss + swap_loss, "loss": clean_loss,
                     "acc": -metrics["accuracy"], "auc": -metrics["auc"]}[args.select]
            thresholds = np.linspace(0.05, 0.95, 91)
            accuracies = np.array([((probabilities >= threshold) == truth).mean() for threshold in thresholds])
            threshold = float(np.median(thresholds[accuracies >= accuracies.max() - 1e-9]))
            return probabilities, clean_loss, swap_loss, score, metrics, threshold

        ema_p, ema_clean, ema_swap, ema_score, ema_metrics, ema_threshold = validate(ema_model)
        raw_p, raw_clean, raw_swap, raw_score, raw_metrics, raw_threshold = validate(raw_model)
        if (not math.isfinite(epoch_loss) or not math.isfinite(ema_score)
                or not math.isfinite(raw_score)):
            raise FloatingPointError("Non-finite loss; no completed checkpoint will be marked")

        ema_improved = ema_score < best_ema_score - 1e-6
        raw_improved = raw_score < best_raw_score - 1e-6
        history.append(dict(
            epoch=epoch, train_loss=epoch_loss / train.G,
            ema_val_loss=ema_clean, ema_val_swap_loss=ema_swap,
            ema_selection_score=ema_score, ema_metrics=ema_metrics,
            raw_val_loss=raw_clean, raw_val_swap_loss=raw_swap,
            raw_selection_score=raw_score, raw_metrics=raw_metrics,
            ema_improved=ema_improved, raw_improved=raw_improved,
        ))

        # Clone both states before saving.  The counterpart state is retained
        # only as provenance; `state` is exactly the weight family being tested.
        current_raw = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        current_ema = {k: v.detach().cpu().clone() for k, v in ema_model.state_dict().items()}
        common = dict(args=saved_args, R=train.R, gm=gm, gs=gs,
                      method=f"v3_{variant}", struct_mode=args.struct_mode,
                      training_complete=False, run_signature=run_signature,
                      provenance=provenance, dual_selection=True)
        if ema_improved:
            best_ema_score, best_ema_epoch = ema_score, epoch
            atomic_save(dict(
                state=current_ema, raw_state=current_raw, ema_state=current_ema,
                human95_threshold=float(np.quantile(ema_p[truth == 1], 0.05, method="lower")), thr=ema_threshold, best_epoch=epoch, weight_source="ema",
                selection_model="ema", selection_score=ema_score, **common,
            ), ema_checkpoint_path)
        if raw_improved:
            best_raw_score, best_raw_epoch = raw_score, epoch
            atomic_save(dict(
                state=current_raw, raw_state=current_raw, ema_state=current_ema,
                human95_threshold=float(np.quantile(raw_p[truth == 1], 0.05, method="lower")), thr=raw_threshold, best_epoch=epoch, weight_source="raw",
                selection_model="raw", selection_score=raw_score, **common,
            ), raw_checkpoint_path)
        bad = 0 if (ema_improved or raw_improved) else bad + 1
        write_json(record_path, dict(
            complete=False, run_signature=run_signature, provenance=provenance,
            best_ema_epoch=best_ema_epoch, best_raw_epoch=best_raw_epoch,
            best_ema_score=best_ema_score, best_raw_score=best_raw_score,
            history=history,
        ))
        print(f"Epoch {epoch:02d} | loss {epoch_loss/train.G:.4f} | "
              f"EMA val {ema_clean:.4f} ({'best' if ema_improved else 'keep'}) | "
              f"Raw val {raw_clean:.4f} ({'best' if raw_improved else 'keep'})", flush=True)
        if bad >= args.patience:
            break

    for checkpoint_path in (ema_checkpoint_path, raw_checkpoint_path):
        finished = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        finished["training_complete"] = True
        atomic_save(finished, checkpoint_path)
    write_json(record_path, dict(
        complete=True, run_signature=run_signature, provenance=provenance,
        checkpoint_sha256={"ema": sha256(ema_checkpoint_path), "raw": sha256(raw_checkpoint_path)},
        best_ema_epoch=best_ema_epoch, best_raw_epoch=best_raw_epoch,
        best_ema_score=best_ema_score, best_raw_score=best_raw_score,
        selection=args.select, history=history, train_seconds=time.time() - start,
    ))
    print(f"Saved EMA: {ema_checkpoint_path}", flush=True)
    print(f"Saved Raw: {raw_checkpoint_path}", flush=True)
    del model, teacher, optimizer, scaler, schedule
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def main(argv=None):
    args = build_parser().parse_args(argv)
    validate_args(args)
    args.determinism = configure_determinism(args.deterministic)
    import torch

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU, or explicitly use --device cpu")
    args.graph_dir, args.output_dir = args.graph_dir.resolve(), args.output_dir.resolve()
    args.output_dir.mkdir(parents=True, exist_ok=True)
    domain = next(item for item in TRAINING_DOMAINS.values()
                  if (item["train_file"], item["val_file"]) == (args.train_file, args.val_file))
    sources = [REPO / domain["train_source"], REPO / domain["val_source"]]
    graphs = [args.graph_dir / f"{name}.pkl" for name in (args.train_file, args.val_file)]
    data_hashes = {str(p): sha256(p) for p in sources + graphs}
    pending = []
    for variant in args.variants:
        for seed in args.seeds:
            paths = {w: args.output_dir / f"v3_{variant}_{w}_seed{seed}.pt" for w in ("raw", "ema")}
            record_path = args.output_dir / f"v3_{variant}_dual_seed{seed}.train.json"
            sig, provenance = signature(args, variant, seed, data_hashes)
            if args.skip_completed and completed_run(paths, record_path, sig):
                print(f"Reuse completed RAW and EMA checkpoints for seed {seed}", flush=True)
            elif (any(path.exists() for path in paths.values()) or record_path.exists()) and not args.overwrite:
                raise FileExistsError(f"Incomplete/different dual run: {record_path}. Use a new output directory "
                                      "or --overwrite; a saved best epoch is not proof training finished.")
            else:
                pending.append((variant, seed, sig, provenance))
    if not pending:
        return
    train = load_graph(graphs[0], sources[0], args.num_rel, args.struct_mode)
    val = load_graph(graphs[1], sources[1], train.R, args.struct_mode)
    for variant, seed, sig, provenance in pending:
        train_one(args, variant, seed, train, val, sig, provenance)


if __name__ == "__main__":
    main()


In [ ]:
from transformers import AutoConfig
from evaluate_v3_quang import sha256
from v3_a1_fusion import source_digest

started = perf_counter()
timings = {}
TRAIN = REPO / 'data/original_text/benchmarks/hc3/hc3_train.json'
VAL = REPO / 'data/original_text/benchmarks/hc3/hc3_val.json'
MODEL_DIR = WORK / 'models'
MODEL_DIR.mkdir(exist_ok=True)
FEATURE_CACHE.mkdir(parents=True, exist_ok=True)
if CACHE_INPUT is not None:
    CACHE_INPUT = Path(CACHE_INPUT)
    if not CACHE_INPUT.is_dir():
        raise FileNotFoundError(CACHE_INPUT)

# Các seed dùng chung cache khi dữ liệu và cách dựng graph trùng khớp.
roberta_revision = AutoConfig.from_pretrained('roberta-base')._commit_hash
if not roberta_revision:
    raise ValueError('Cannot identify the RoBERTa revision for the feature cache')
feature_versions = {p: version(p) for p in ('torch', 'transformers', 'tokenizers', 'spacy', 'en-core-web-sm', 'torch-geometric')}
feature_versions.update(roberta=roberta_revision, vocab=sha256(REPO / 'dep_vocab.json'),
                        vocab_code=sha256(REPO / 'dep_vocab.py'))
local_encoder = REPO / 'roberta-base'
v3_encoder = {p.name: sha256(p) for p in local_encoder.iterdir() if p.is_file()} if (local_encoder / 'vocab.json').is_file() else roberta_revision

def feature_dir(branch, sources, recipe, names):
    values = [feature_versions, recipe, [(s.stem, source_digest(s)) for s in sources]]
    key = hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()
    folder = f'{branch}-{key}'
    for root in (FEATURE_CACHE, CACHE_INPUT):
        if root is not None and all((root / folder / n).is_file() for n in names):
            print(f'Cache hit: {branch} / {", ".join(names)}', flush=True)
            return root / folder, True
    dest = FEATURE_CACHE / folder
    dest.mkdir(exist_ok=True)
    return dest, False

def v3_graphs(sources):
    names = [s.stem + '.pkl' for s in sources]
    dest, hit = feature_dir('v3', sources, [sha256(REPO / 'building_graph.py'), v3_encoder], names)
    if not hit:
        with TemporaryDirectory(dir=dest) as stage:
            stage = Path(stage)
            if (local_encoder / 'vocab.json').is_file():
                (stage / 'roberta-base').symlink_to(local_encoder, target_is_directory=True)
            subprocess.run([sys.executable, str(REPO / 'building_graph.py'), '-f', *map(str, sources)], cwd=stage, check=True)
            for name in names:
                (stage / 'graph_data' / name).replace(dest / name)
    return dest

stage_start = perf_counter()
v3_train_dir = v3_graphs([TRAIN, VAL])
timings['v3_train_graphs_seconds'] = perf_counter() - stage_start
v3_command = [sys.executable, 'train_v3_dual_best.py', '--seeds', str(SEED), '--device', 'cuda',
              '--no-deterministic', '--ema', '1', '--epochs', str(V3_EPOCHS),
              '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE),
              '--val-batch-size', str(VAL_BATCH_SIZE), '--graph-dir', str(v3_train_dir),
              '--output-dir', str(MODEL_DIR), '--skip-completed']
V3_CHECKPOINTS = {w: MODEL_DIR / f'v3_baseline_{w}_seed{SEED}.pt' for w in ('raw', 'ema')}
V3_RECORD = MODEL_DIR / f'v3_baseline_dual_seed{SEED}.train.json'
parallel = PARALLEL_IF_TWO_GPUS and torch.cuda.device_count() >= 2
if not parallel:
    subprocess.run(v3_command, check=True)
print('Train schedule:', 'A1 on GPU 0 and V3 on GPU 1' if parallel else 'V3 then A1')


## Tạo embedding A1 và huấn luyện R-GCN

RoBERTa đóng băng. Token spaCy được căn chỉnh với subword, lấy trung bình subword và các cửa sổ chồng lấp. Chọn checkpoint A1 bằng validation BCE loss. Cache được lưu theo split để có thể kiểm tra hoặc tái sử dụng.


In [ ]:
import inspect
from transformers import AutoTokenizer, AutoModel
stage_start = perf_counter()
import spacy
from dep_vocab import get_default_vocab
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from model.RGCN import RGCN2
from v3_a1_fusion import read_source, labels_from, source_digest

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
nlp = spacy.load('en_core_web_sm', disable=['ner'])
tokenizer = AutoTokenizer.from_pretrained('roberta-base', revision=roberta_revision, use_fast=True, add_prefix_space=True)
encoder = AutoModel.from_pretrained('roberta-base', revision=roberta_revision, add_pooling_layer=False).cuda().eval()
vocab = get_default_vocab()
def make_graphs(source):
    rows = read_source(source)
    labels = labels_from(rows)
    graphs = []
    for i, doc in enumerate(nlp.pipe((r['text'] for r in rows), batch_size=16)):
        if not len(doc):
            raise ValueError(f'Empty document {i} in {source}')
        tokens = tokenizer([w.text for w in doc], is_split_into_words=True, max_length=512,
                           stride=64, truncation=True, return_overflowing_tokens=True,
                           padding=True, return_tensors='pt')
        sums = torch.zeros((len(doc), 768), device='cuda')
        counts = torch.zeros(len(doc), device='cuda')
        with torch.inference_mode():
            for window in range(len(tokens['input_ids'])):
                size = int(tokens['attention_mask'][window].sum())
                ids = tokens['input_ids'][window:window+1, :size].cuda()
                mask = tokens['attention_mask'][window:window+1, :size].cuda()
                with torch.autocast('cuda'):
                    hidden = encoder(input_ids=ids, attention_mask=mask).last_hidden_state[0].float()
                mapping = tokens.word_ids(window)[:size]
                positions = [j for j, word in enumerate(mapping) if word is not None]
                words = torch.tensor([mapping[j] for j in positions], device='cuda')
                sums.index_add_(0, words, hidden[positions])
                counts.index_add_(0, words, torch.ones_like(words, dtype=torch.float))
        x = (sums / counts.clamp_min(1).unsqueeze(1)).cpu().half()
        edge = torch.tensor([[w.i for w in doc], [w.head.i for w in doc]], dtype=torch.long)
        rel = torch.tensor([vocab.encode(w.dep_) for w in doc], dtype=torch.long)
        if int(rel.max()) >= 60: raise ValueError('A1 relation ID exceeds 59')
        graphs.append(Data(x=x, edge_index=edge, edge_type=rel, y=torch.tensor([labels[i]], dtype=torch.float)))
        if (i+1) % 500 == 0: print(source.stem, i+1, '/', len(rows), flush=True)
    return graphs

graph_files = {}
recipe = inspect.getsource(make_graphs)
for source in (TRAIN, VAL):
    name = source.stem + '.pt'
    folder, hit = feature_dir('a1', [source], recipe, [name])
    graph_files[source.stem] = folder / name
    if not hit:
        graphs = make_graphs(source)
        with TemporaryDirectory(dir=folder) as stage:
            tmp = Path(stage) / name
            torch.save({'source_sha256': source_digest(source), 'graphs': graphs}, tmp)
            tmp.replace(folder / name)
        del graphs
del encoder, tokenizer, nlp
torch.cuda.empty_cache()
timings['a1_train_graphs_seconds'] = perf_counter() - stage_start


In [ ]:
A1_CHECKPOINT = MODEL_DIR / f'a1_seed{SEED}.pt'

def train_a1():
    train_cache = torch.load(graph_files[TRAIN.stem], weights_only=False)
    val_cache = torch.load(graph_files[VAL.stem], weights_only=False)
    assert train_cache['source_sha256'] == source_digest(TRAIN)
    assert val_cache['source_sha256'] == source_digest(VAL)
    train_loader = DataLoader(train_cache['graphs'], batch_size=A1_BATCH, shuffle=True)
    val_loader = DataLoader(val_cache['graphs'], batch_size=A1_BATCH, shuffle=False)
    model = RGCN2(input_dim=768, hidden_dim=256, output_dim=64, num_relations=60, num_bases=30).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=A1_LR)
    best = float('inf')
    history = []
    for epoch in range(1, A1_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            batch = batch.cuda(); batch.x = batch.x.float()
            optimizer.zero_grad(set_to_none=True)
            loss = torch.nn.functional.binary_cross_entropy(model(batch).flatten(), batch.y)
            loss.backward(); optimizer.step()
        model.eval(); losses = []; count = 0
        with torch.inference_mode():
            for batch in val_loader:
                batch = batch.cuda(); batch.x = batch.x.float()
                losses.append(torch.nn.functional.binary_cross_entropy(model(batch).flatten(), batch.y, reduction='sum'))
                count += batch.num_graphs
        val_loss = sum(torch.stack(losses).tolist()) / count
        history.append({'epoch':epoch, 'validation_bce':val_loss})
        if val_loss < best:
            best = val_loss
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, A1_CHECKPOINT)
        print(f'A1 epoch {epoch}: validation BCE {val_loss:.5f}', flush=True)
    (WORK / 'a1_history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

stage_start = perf_counter()
if parallel:
    visible = os.environ.get('CUDA_VISIBLE_DEVICES')
    second = visible.split(',')[1].strip() if visible else '1'
    with subprocess.Popen(v3_command, env={**os.environ, 'CUDA_VISIBLE_DEVICES': second}) as v3_job:
        try:
            train_a1()
            timings['a1_train_seconds'] = perf_counter() - stage_start
            if v3_job.wait() != 0:
                raise subprocess.CalledProcessError(v3_job.returncode, v3_command)
        finally:
            if v3_job.poll() is None:
                v3_job.terminate()
else:
    train_a1()
    timings['a1_train_seconds'] = perf_counter() - stage_start
torch.cuda.empty_cache()
v3_record = json.loads(V3_RECORD.read_text(encoding='utf-8'))
if not v3_record['complete'] or v3_record['checkpoint_sha256'] != {w: sha256(p) for w, p in V3_CHECKPOINTS.items()}:
    raise ValueError('Dual training record does not match the completed RAW and EMA files')
timings['v3_checkpoint_train_seconds'] = v3_record['train_seconds']
print('Best HC3 validation epochs:', {w: v3_record[f'best_{w}_epoch'] for w in V3_CHECKPOINTS})
print('Seconds:', timings)


## So sánh trên ba tập test

Mỗi tập dùng chung một graph V3 và một bộ điểm A1. Bảng cuối hiển thị accuracy ở ngưỡng 0,5 và ở ngưỡng human95 riêng của từng checkpoint V3. Hai cột kết hợp đều dùng ngưỡng 0,5 trên tích điểm.

Bộ chọn nhánh học từ HC3 train và các bản thay từ nhân tạo. Ngưỡng chọn nhánh lấy từ phân vị 95% của điểm nhiễu trên HC3 validation sạch.


In [ ]:
%%writefile v3_a1_routing.py
"""Select A1 or V3 for one English text using a corruption detector fitted on HC3."""

from collections import Counter
import hashlib
import math
import re

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


_WORD = re.compile(r"[a-z]+(?:'[a-z]+)?", re.IGNORECASE | re.ASCII)
_FEATURES = ("mean_word_surprisal", "unknown_fraction", "mean_bigram_association", "unseen_pair_fraction",
             "mean_context_surprisal", "open_class_fraction")


def _contexts(tokens, common):
    classes = ["<boundary>", *(w if w in common else "<other>" for w in tokens), "<boundary>"]
    return zip(classes, classes[1:], classes[2:])


def _tokens(text):
    return _WORD.findall(text.lower().replace("’", "'"))


def _text_key(text):
    return hashlib.sha256(" ".join(text.split()).encode()).digest()


def _features(tokens, reference):
    words, pairs, total, common, centers, windows, triples = reference
    if not tokens:
        return (math.log(total + 1), 1.0, 0.0, 1.0, 0.0, 1.0)
    counts = [words[w] for w in tokens]
    rarity = math.log(total + 1) - sum(math.log1p(c) for c in counts) / len(tokens)
    unknown = counts.count(0) / len(tokens)
    association = 0.0
    unseen = 0
    # Compare each transition to its unigram background with bigram backoff.
    for i, pair in enumerate(zip(tokens, tokens[1:])):
        hits = pairs[pair]
        unseen += hits == 0
        left, right = counts[i], counts[i + 1]
        for context, target in ((left, right), (right, left)):
            association += math.log1p(hits * total / (10 * (target + 1))) - math.log1p(context / 10)
    n_pairs = max(1, len(tokens) - 1)
    surprise = 0.0
    for left, center, right in _contexts(tokens, common):
        prior = (centers[center] + 1) / (total + len(centers) + 1)
        likelihood = (triples[left, center, right] + 10 * prior) / (windows[left, right] + 10)
        surprise -= math.log(likelihood)
    other = sum(w not in common for w in tokens)
    return rarity, unknown, association / (2 * n_pairs), unseen / n_pairs, surprise / len(tokens), other / len(tokens)


def _replace(tokens, vocabulary, rng):
    changed = list(tokens)
    if changed:
        n = max(1, round(len(tokens) * rng.uniform(0.10, 0.35)))
        positions = rng.choice(len(tokens), size=n, replace=False)
        for idx, donor in zip(positions, rng.integers(len(vocabulary), size=n)):
            changed[idx] = vocabulary[donor]
    return changed


def fit_router(train_texts, validation_texts):
    """Training text supplies reference counts and synthetic edits. Validation sets the cutoff."""
    train_texts = list(train_texts)
    keys = {_text_key(text) for text in train_texts}
    words, pairs = Counter(), Counter()
    examples, reference_docs = [], []
    for text in train_texts:
        tokens = _tokens(text)
        if int.from_bytes(_text_key(text)[:4], "big") % 5 == 0:
            examples.append(tokens)
        else:
            words.update(tokens)
            pairs.update(zip(tokens, tokens[1:]))
            reference_docs.append(tokens)
    common = {w for w, _ in words.most_common(256)}
    centers, windows, triples = Counter(), Counter(), Counter()
    for tokens in reference_docs:
        for left, center, right in _contexts(tokens, common):
            centers[center] += 1
            windows[left, right] += 1
            triples[left, center, right] += 1
    reference = (words, pairs, sum(words.values()), common, centers, windows, triples)
    donors = sorted({word for tokens in examples for word in tokens})
    if not donors or not reference[2]:
        raise ValueError("Need documents in both parts of the source training split")
    rng = np.random.default_rng(42)
    pristine = [_features(tokens, reference) for tokens in examples]
    edited = [_features(_replace(tokens, donors, rng), reference) for tokens in examples]
    detector = make_pipeline(StandardScaler(), LogisticRegression())
    detector.fit(pristine + edited, [0] * len(pristine) + [1] * len(edited))
    validation = [text for text in validation_texts if _text_key(text) not in keys]
    if not validation:
        raise ValueError("Need clean source validation documents separate from training")
    clean_scores = detector.decision_function([_features(_tokens(t), reference) for t in validation])
    cutoff = float(np.quantile(clean_scores, 0.95, method="higher"))
    edited_scores = detector.decision_function([
        _features(_replace(_tokens(t), donors, rng), reference) for t in validation])
    return {"reference": reference, "detector": detector, "cutoff": cutoff,
            "source_validation": {"documents": len(validation), "features": list(_FEATURES),
                                  "clean_routed_to_v3": float(np.mean(clean_scores > cutoff)),
                                  "synthetic_routed_to_v3": float(np.mean(edited_scores > cutoff))}}


def use_v3(router, texts):
    features = [_features(_tokens(t), router["reference"]) for t in texts]
    if not features:
        return np.empty(0, dtype=bool)
    return router["detector"].decision_function(features) > router["cutoff"]


def predict_human(route_to_v3, v3_p_human, a1_logit_human, human95_threshold):
    mask = np.asarray(route_to_v3, dtype=bool)
    p, z = np.asarray(v3_p_human, dtype=float), np.asarray(a1_logit_human, dtype=float)
    if (mask.shape != p.shape or z.shape != p.shape or not np.isfinite(z).all()
            or not np.all((p >= 0) & (p <= 1)) or not 0 <= human95_threshold <= 1):
        raise ValueError("Expected aligned decisions, probabilities, logits and a valid V3 threshold")
    return np.where(mask, p >= human95_threshold, z >= 0)


In [ ]:
from types import SimpleNamespace
import evaluate_v3_quang as ev
from v3_a1_fusion import paired_scores
from v3_a1_routing import fit_router, predict_human, use_v3
import joblib

router_start = perf_counter()
router = fit_router([r['text'] for r in read_source(TRAIN)], [r['text'] for r in read_source(VAL)])
router_path = MODEL_DIR / 'hc3_router.joblib'
joblib.dump(router, router_path, compress=3)
timings['router_fit_seconds'] = perf_counter() - router_start

cases = [
    ('hc3', 'data/original_text/benchmarks/hc3/hc3_test.json'),
    ('gpt35_0pct', 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json'),
    ('hc3_replace25', 'data/perturbed_text/hc3/structural_noise/hc3_test_replace_25.json'),
]
stage_start = perf_counter()
OUT = WORK / 'results'
OUT.mkdir(exist_ok=True)
from scipy.special import expit

configs = {}
for weight in V3_CHECKPOINTS:
    args = SimpleNamespace(variant='baseline', train_domain='hc3', checkpoint_dir=str(MODEL_DIR),
                           seeds=[SEED], batch_size=1, struct_mode='low', device='cuda',
                           graph_budget_root=str(WORK), weights=weight)
    path = OUT / ('v3_config.json' if weight == 'raw' else 'v3_ema_config.json')
    ev.write_json(path, ev.make_config(args))
    configs[weight] = path

a1_digest = ev.sha256(A1_CHECKPOINT)
subprocess.run([sys.executable, 'v3_a1_fusion.py', 'score-a1', '--sources', *[str(REPO / rel) for _, rel in cases],
                '--checkpoint', str(A1_CHECKPOINT), '--device', 'cuda', '--output-dir', str(OUT)], check=True)
accuracy = {}
thresholds = {}
for name, relative in cases:
    source = REPO / relative
    rows = read_source(source)
    truth = labels_from(rows)
    graph = v3_graphs([source]) / f'{source.stem}.pkl'
    a1 = ev.read_json(OUT / f'scores_{source.stem}.json')
    if a1['checkpoint_sha256'] != a1_digest:
        raise ValueError(f'A1 checkpoint mismatch for {name}')
    a1_logits = np.array([r['logit_human'] for r in a1['records']])
    a1_p = expit(a1_logits)
    a1_metrics = ev.score_probabilities(truth, a1_p, 0.5)
    row = {'A1@0.5': a1_metrics['accuracy']}
    for weight, config_path in configs.items():
        # Existing RAW report names are also consumed by the V3+A1 notebooks.
        prefix = name if weight == 'raw' else f'{name}_ema'
        v3_file = OUT / f'{prefix}_v3.json'
        eval_start = perf_counter()
        subprocess.run([sys.executable, 'evaluate_v3_quang.py', '_test', '--config', str(config_path),
                        '--seed', str(SEED), '--graph', str(graph), '--expected-samples', str(len(rows)),
                        '--signature', f'{name}_{weight}_seed{SEED}', '--report', str(v3_file)], check=True)
        timing_key = f'v3_{weight}_test_seconds'
        timings[timing_key] = timings.get(timing_key, 0) + perf_counter() - eval_start
        v3 = ev.read_json(v3_file)
        if v3['checkpoint_sha256'] != v3_record['checkpoint_sha256'][weight]:
            raise ValueError(f'Unexpected {weight} checkpoint for {name}')
        combined = paired_scores(source, truth, v3, a1)
        fusion = ev.score_probabilities(truth, combined, 0.5)
        human95 = v3['at_source_human95']
        thresholds[weight] = v3['source_human95_threshold']
        result = {'seed': SEED, 'source': relative, 'samples': len(rows), 'threshold': 0.5,
                  'weight_source': weight, 'v3_sha256': v3['checkpoint_sha256'], 'a1_sha256': a1_digest,
                  'v3': ev.score_probabilities(truth, np.asarray(v3['p_human']), 0.5),
                  'source_human95_threshold': v3['source_human95_threshold'], 'v3_at_source_human95': human95,
                  'a1': a1_metrics, 'fusion': fusion, 'y_true': truth.tolist(), 'p_human_fusion': combined.tolist()}
        if weight == 'raw':
            route = use_v3(router, [r['text'] for r in rows])
            predicted = predict_human(route, v3['p_human'], a1_logits, thresholds[weight])
            row['A1/V3 routed'] = float((predicted == truth).mean())
            result['routed'] = {'accuracy': row['A1/V3 routed'], 'use_v3': route.tolist(),
                                'predicted_human': predicted.astype(int).tolist()}
        ev.write_json(OUT / f'{prefix}_fusion.json', result)
        label = weight.upper()
        row[f'{label}@0.5'] = v3['accuracy']
        row[f'{label}@human95'] = human95['accuracy']
        row[f'{label}+A1@0.5'] = fusion['accuracy']
    accuracy[name] = row

print('HC3 validation human95 thresholds:', thresholds)
columns = ['RAW@0.5', 'EMA@0.5', 'RAW@human95', 'EMA@human95', 'A1@0.5', 'RAW+A1@0.5', 'EMA+A1@0.5', 'A1/V3 routed']
print('Test set'.ljust(16), *(c.rjust(12) for c in columns))
for name, row in accuracy.items():
    print(name.ljust(16), *(f'{row[c]:12.2%}' for c in columns))
ev.write_json(OUT / 'summary.json', {'seed': SEED, 'accuracy': accuracy, 'source_human95_thresholds': thresholds})
timings['three_tests_seconds'] = perf_counter() - stage_start


In [ ]:
manifest = {'seed':SEED, 'v3_epochs':V3_EPOCHS, 'a1_epochs':A1_EPOCHS,
            'v3_batch_size':BATCH_SIZE, 'v3_micro_batch_size':MICRO_BATCH_SIZE, 'v3_val_batch_size':VAL_BATCH_SIZE,
            'a1_lr':A1_LR, 'a1_batch':A1_BATCH, 'commit':subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(),
            'v3_sha256':v3_record['checkpoint_sha256']['raw'], 'v3_ema_sha256':v3_record['checkpoint_sha256']['ema'],
            'a1_sha256':a1_digest, 'best_v3_epochs':{w:v3_record[f'best_{w}_epoch'] for w in V3_CHECKPOINTS}}
timings['wall_seconds_graphs_train_tests'] = perf_counter() - started
manifest.update(timings=timings, parallel_training=parallel, router_sha256=ev.sha256(router_path),
                router_code_sha256=ev.sha256(REPO / 'v3_a1_routing.py'), router_cutoff=router['cutoff'],
                router_validation=router['source_validation'])
ev.write_json(OUT / 'manifest.json', manifest)
print('Minutes:', {key: round(value / 60, 1) for key, value in timings.items()})
archive = ROOT / f'v3_dual_a1_hc3_seed{SEED}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in list(OUT.glob('*.json')) + [*V3_CHECKPOINTS.values(), A1_CHECKPOINT, router_path, V3_RECORD, WORK / 'a1_history.json']:
        z.write(path, path.relative_to(WORK))
    z.write(REPO / 'train_v3_dual_best.py', 'train_v3_dual_best.py')
    z.write(REPO / 'v3_a1_routing.py', 'v3_a1_routing.py')
print('Download:', archive)
